In [ ]:
# ============================================================
# GPR-SHAP Analysis of Aging Index
# Standalone version
#
# Required files:
#   /content/Aging index.xlsx
#   /content/GPR_AgingIndex_FBmass_LOMO_final.pkl
# ============================================================

# If needed in Colab:
# !pip install shap openpyxl

import joblib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker

from matplotlib.colors import LinearSegmentedColormap

import shap

warnings.filterwarnings("ignore")


# ============================================================
# 1. FILE SETTINGS
# ============================================================

DATA_FILE = "/content/Aging index.xlsx"

MODEL_FILE = (
    "/content/GPR_AgingIndex_FBmass_LOMO_final.pkl"
)


# ============================================================
# 2. LOAD TRAINED GPR MODEL
# ============================================================

gpr_framework = joblib.load(
    MODEL_FILE
)

best_final_model = gpr_framework[
    "model"
]

print("GPR model loaded successfully.")

print("\nModel validation:")
print(
    gpr_framework.get(
        "validation",
        "Not available"
    )
)

print("\nBest parameters:")
print(
    gpr_framework.get(
        "best_params",
        "Not available"
    )
)


# ============================================================
# 3. LOAD AGING INDEX DATA
# ============================================================

data = pd.read_excel(
    DATA_FILE
)

data.columns = (
    data.columns
    .astype(str)
    .str.strip()
)

required_columns = [
    "Filler type",
    "F/B by weight",
    "Frequency (rad/s)",
    "Aging Index"
]

missing_columns = [
    column
    for column in required_columns
    if column not in data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {data.columns.tolist()}"
    )

data = data[
    required_columns
].copy()

data = data.rename(
    columns={
        "Filler type": "Filler_type",
        "F/B by weight": "FB_mass",
        "Frequency (rad/s)": "Frequency",
        "Aging Index": "Aging_Index"
    }
)

data["Filler_type"] = (
    data["Filler_type"]
    .astype(str)
    .str.strip()
)

for column in [
    "FB_mass",
    "Frequency",
    "Aging_Index"
]:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )

data = (
    data
    .dropna()
    .reset_index(drop=True)
)

data = data[
    (data["Frequency"] > 0)
    & (data["Aging_Index"] > 0)
].reset_index(drop=True)


# ============================================================
# 4. STANDARDIZE FILLER NAMES
# ============================================================

data["Filler_type"] = (
    data["Filler_type"]
    .replace({
        "Rh": "Rheofiller",
        "RH": "Rheofiller",
        "rh": "Rheofiller",
        "rheofiller": "Rheofiller",
        "RHEOFILLER": "Rheofiller",

        "Ce": "Cement",
        "CE": "Cement",
        "ce": "Cement",
        "cement": "Cement",
        "CEMENT": "Cement",

        "Ch": "Chromium steel sludge",
        "CH": "Chromium steel sludge",
        "ch": "Chromium steel sludge",
        "Chromium Steel Sludge":
            "Chromium steel sludge",
        "chromium steel sludge":
            "Chromium steel sludge"
    })
)

data["log10_Frequency"] = np.log10(
    data["Frequency"]
)

data["Mastic_Group"] = (
    data["Filler_type"].astype(str)
    + "_FBmass_"
    + data["FB_mass"].round(3).astype(str)
)

print("\nFiller types:")
print(
    data["Filler_type"].unique()
)

print("\nNumber of observations:")
print(len(data))

print("\nNumber of mastic groups:")
print(
    data["Mastic_Group"].nunique()
)


# ============================================================
# 5. FIGURE STYLE
# ============================================================

try:
    font_path = "/content/TimesNewRoman.ttf"

    fm.fontManager.addfont(
        font_path
    )

    font_name = fm.FontProperties(
        fname=font_path
    ).get_name()

    mpl.rcParams[
        "font.family"
    ] = font_name

except Exception:
    mpl.rcParams[
        "font.family"
    ] = "Times New Roman"


mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8


red_to_black = (
    LinearSegmentedColormap.from_list(
        "RedBlack",
        [
            "#FF0000",
            "#000000"
        ]
    )
)


# ============================================================
# 6. PREPARE FILLER NAMES FOR SHAP
# ============================================================

# The fitted GPR model was trained with
# "Chromium steel sludge".
# For the SHAP display, a shorter label is used.

filler_to_code = {
    "Chromium steel sludge": 0,
    "Cement": 1,
    "Rheofiller": 2
}

code_to_filler = {
    0: "Chromium steel sludge",
    1: "Cement",
    2: "Rheofiller"
}


# ============================================================
# 7. PREPARE SHAP INPUTS
# ============================================================

X_shap = pd.DataFrame({
    "Filler type":
        data["Filler_type"].map(
            filler_to_code
        ),

    "f/b by weight":
        data["FB_mass"],

    "log10 frequency":
        data["log10_Frequency"]
})


if X_shap.isna().any().any():

    unknown_fillers = data.loc[
        X_shap["Filler type"].isna(),
        "Filler_type"
    ].unique()

    raise ValueError(
        "Unrecognized filler names: "
        f"{unknown_fillers}"
    )


X_shap = X_shap.astype(float)

X_display = X_shap.copy()

X_display.columns = [
    "Filler type",
    "f/b by weight",
    r"$\log_{10}$ frequency"
]

print("\nSHAP input data:")
display(
    X_display.head()
)


# ============================================================
# 8. SHAP PREDICTION WRAPPER
# ============================================================

def predict_aging_index(arr):

    arr = np.asarray(
        arr,
        dtype=float
    )

    if arr.ndim == 1:
        arr = arr.reshape(
            1,
            -1
        )

    filler_codes = np.rint(
        arr[:, 0]
    ).astype(int)

    filler_codes = np.clip(
        filler_codes,
        0,
        2
    )

    prediction_df = pd.DataFrame({
        "Filler_type": [
            code_to_filler[code]
            for code in filler_codes
        ],

        "FB_mass":
            arr[:, 1],

        "log10_Frequency":
            arr[:, 2]
    })

    return best_final_model.predict(
        prediction_df
    )


# ============================================================
# 9. VERIFY PREDICTION WRAPPER
# ============================================================

X_original = data[
    [
        "Filler_type",
        "FB_mass",
        "log10_Frequency"
    ]
].copy()

direct_predictions = (
    best_final_model.predict(
        X_original
    )
)

wrapped_predictions = (
    predict_aging_index(
        X_shap.values
    )
)

maximum_difference = np.max(
    np.abs(
        direct_predictions
        - wrapped_predictions
    )
)

print(
    "\nMaximum difference between "
    "direct and SHAP-wrapper predictions:"
)

print(maximum_difference)


if maximum_difference > 1e-8:
    raise ValueError(
        "The SHAP wrapper does not reproduce "
        "the original GPR predictions."
    )


# ============================================================
# 10. KERNEL SHAP
# ============================================================

X_values = X_shap.values

background = X_values

explainer = shap.KernelExplainer(
    predict_aging_index,
    background
)

shap_values = explainer.shap_values(
    X_values,
    nsamples="auto"
)

if isinstance(
    shap_values,
    list
):
    shap_values = shap_values[0]

shap_values = np.asarray(
    shap_values,
    dtype=float
)

if shap_values.ndim == 3:
    shap_values = shap_values[
        :,
        :,
        0
    ]

print("\nSHAP matrix shape:")
print(
    shap_values.shape
)


# ============================================================
# 11. VERIFY SHAP ADDITIVITY
# ============================================================

expected_value = np.asarray(
    explainer.expected_value
).reshape(-1)[0]

reconstructed_predictions = (
    expected_value
    + shap_values.sum(axis=1)
)

maximum_additivity_error = np.max(
    np.abs(
        wrapped_predictions
        - reconstructed_predictions
    )
)

print("\nSHAP expected value:")
print(expected_value)

print(
    "\nMaximum SHAP additivity error:"
)
print(
    maximum_additivity_error
)


# ============================================================
# 12. GLOBAL SHAP IMPORTANCE
# ============================================================

mean_absolute_shap = (
    np.abs(
        shap_values
    ).mean(axis=0)
)

importance_df = pd.DataFrame({
    "Feature":
        X_display.columns,

    "Mean |SHAP|":
        mean_absolute_shap
})

importance_df = (
    importance_df
    .sort_values(
        "Mean |SHAP|",
        ascending=False
    )
    .reset_index(drop=True)
)

importance_df[
    "Relative importance (%)"
] = (
    importance_df["Mean |SHAP|"]
    / importance_df["Mean |SHAP|"].sum()
    * 100
)

print(
    "\nGlobal SHAP importance:"
)

display(
    importance_df
)


# ============================================================
# 13. SAVE SHAP VALUES
# ============================================================

shap_values_df = pd.DataFrame(
    shap_values,
    columns=[
        "SHAP Filler type",
        "SHAP f/b by weight",
        "SHAP log10 frequency"
    ]
)

shap_results = data[
    [
        "Filler_type",
        "FB_mass",
        "Frequency",
        "log10_Frequency",
        "Aging_Index",
        "Mastic_Group"
    ]
].copy()

shap_results[
    "Predicted Aging Index"
] = wrapped_predictions

shap_results[
    "SHAP Filler type"
] = shap_values[:, 0]

shap_results[
    "SHAP f/b by weight"
] = shap_values[:, 1]

shap_results[
    "SHAP log10 frequency"
] = shap_values[:, 2]

shap_results[
    "SHAP base value"
] = expected_value

shap_results[
    "SHAP reconstructed prediction"
] = reconstructed_predictions


filler_summary = (
    shap_results
    .groupby("Filler_type")
    .agg(
        Mean_SHAP=(
            "SHAP Filler type",
            "mean"
        ),

        Mean_absolute_SHAP=(
            "SHAP Filler type",
            lambda values: np.mean(
                np.abs(values)
            )
        ),

        Minimum_SHAP=(
            "SHAP Filler type",
            "min"
        ),

        Maximum_SHAP=(
            "SHAP Filler type",
            "max"
        )
    )
    .reset_index()
)


with pd.ExcelWriter(
    "GPR_AgingIndex_SHAP_outputs.xlsx",
    engine="openpyxl"
) as writer:

    shap_results.to_excel(
        writer,
        sheet_name="SHAP Results",
        index=False
    )

    shap_values_df.to_excel(
        writer,
        sheet_name="SHAP Values",
        index=False
    )

    importance_df.to_excel(
        writer,
        sheet_name="Global Importance",
        index=False
    )

    filler_summary.to_excel(
        writer,
        sheet_name="Filler Summary",
        index=False
    )


# ============================================================
# 14. CUSTOM SHAP SUMMARY PLOT
# ============================================================

def shap_summary_template(
    ax,
    shap_values,
    X_df,
    title_text
):

    mean_abs = np.abs(
        shap_values
    ).mean(axis=0)

    order = np.argsort(
        mean_abs
    )[::-1]

    shap_sorted = shap_values[
        :,
        order
    ]

    values_sorted = X_df.values[
        :,
        order
    ]

    feature_sorted = [
        X_df.columns[i]
        for i in order
    ]

    rng = np.random.default_rng(
        42
    )

    for i in range(
        len(feature_sorted)
    ):

        feature_values = (
            values_sorted[:, i]
        )

        shap_feature_values = (
            shap_sorted[:, i]
        )

        vmin = np.nanmin(
            feature_values
        )

        vmax = np.nanmax(
            feature_values
        )

        if vmax > vmin:

            normalized_values = (
                feature_values
                - vmin
            ) / (
                vmax
                - vmin
            )

        else:

            normalized_values = (
                np.full_like(
                    feature_values,
                    0.05,
                    dtype=float
                )
            )

        y_jitter = rng.normal(
            loc=i,
            scale=0.055,
            size=len(
                shap_feature_values
            )
        )

        scatter = ax.scatter(
            shap_feature_values,
            y_jitter,
            c=normalized_values,
            cmap=red_to_black,
            s=34,
            edgecolors="none",
            alpha=0.88
        )

    ax.set_yticks(
        np.arange(
            len(feature_sorted)
        )
    )

    ax.set_yticklabels(
        feature_sorted,
        fontsize=12
    )

    ax.invert_yaxis()

    ax.set_xlabel(
        "SHAP value "
        "(impact on predicted Aging Index)",
        fontsize=12
    )

    ax.set_title(
        title_text,
        fontsize=13,
        pad=8
    )

    ax.tick_params(
        axis="x",
        labelsize=11
    )

    ax.tick_params(
        axis="y",
        labelsize=12
    )

    xmax = np.nanmax(
        np.abs(
            shap_sorted
        )
    )

    ax.set_xlim(
        -xmax * 1.12,
        xmax * 1.12
    )

    ax.axvline(
        0,
        color="black",
        linestyle="--",
        linewidth=0.8
    )

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.22
    )

    ax.xaxis.set_major_locator(
        mticker.MaxNLocator(6)
    )

    ax.xaxis.set_major_formatter(
        mticker.FormatStrFormatter(
            "%.2f"
        )
    )

    for spine in [
        "top",
        "right"
    ]:
        ax.spines[
            spine
        ].set_visible(False)

    return scatter


# ============================================================
# 15. SHAP SUMMARY FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.8, 4.8)
)

plt.subplots_adjust(
    left=0.24,
    right=0.84,
    bottom=0.18,
    top=0.88
)

scatter = shap_summary_template(
    ax,
    shap_values,
    X_display,
    "GPR prediction of Aging Index"
)

cax = fig.add_axes(
    [
        0.875,
        0.20,
        0.018,
        0.62
    ]
)

colorbar = fig.colorbar(
    scatter,
    cax=cax
)

colorbar.set_ticks(
    [0, 1]
)

colorbar.set_ticklabels(
    [
        "Low",
        "High"
    ]
)

colorbar.outline.set_visible(
    False
)

colorbar.ax.tick_params(
    axis="y",
    length=0,
    labelsize=10
)

colorbar.set_label(
    "Feature value",
    fontsize=11,
    rotation=270,
    labelpad=12
)

plt.savefig(
    "Figure_GPR_AgingIndex_SHAP_summary.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "Figure_GPR_AgingIndex_SHAP_summary.jpeg",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 16. GLOBAL IMPORTANCE BAR PLOT
# ============================================================

importance_plot = (
    importance_df
    .sort_values(
        "Mean |SHAP|",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(7.2, 4.4)
)

ax.barh(
    importance_plot["Feature"],
    importance_plot["Mean |SHAP|"]
)

ax.set_xlabel(
    "Mean absolute SHAP value",
    fontsize=12
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.22
)

for spine in [
    "top",
    "right"
]:
    ax.spines[
        spine
    ].set_visible(False)

plt.tight_layout()

plt.savefig(
    "Figure_GPR_AgingIndex_SHAP_importance.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "Figure_GPR_AgingIndex_SHAP_importance.jpeg",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 17. MEAN SHAP EFFECT OF EACH FILLER
# ============================================================

filler_plot = (
    filler_summary
    .sort_values(
        "Mean_SHAP"
    )
)

fig, ax = plt.subplots(
    figsize=(7.2, 4.4)
)

ax.barh(
    filler_plot["Filler_type"],
    filler_plot["Mean_SHAP"]
)

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=0.8
)

ax.set_xlabel(
    "Mean SHAP value for "
    "predicted Aging Index",
    fontsize=12
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.22
)

for spine in [
    "top",
    "right"
]:
    ax.spines[
        spine
    ].set_visible(False)

plt.tight_layout()

plt.savefig(
    "Figure_GPR_AgingIndex_SHAP_filler.pdf",
    bbox_inches="tight"
)

plt.savefig(
    "Figure_GPR_AgingIndex_SHAP_filler.jpeg",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 18. FINAL OUTPUT
# ============================================================

print(
    "\nSHAP analysis completed successfully."
)

print("\nSaved files:")

print(
    "1. GPR_AgingIndex_SHAP_outputs.xlsx"
)

print(
    "2. Figure_GPR_AgingIndex_SHAP_summary.pdf"
)

print(
    "3. Figure_GPR_AgingIndex_SHAP_summary.jpeg"
)

print(
    "4. Figure_GPR_AgingIndex_SHAP_importance.pdf"
)

print(
    "5. Figure_GPR_AgingIndex_SHAP_importance.jpeg"
)

print(
    "6. Figure_GPR_AgingIndex_SHAP_filler.pdf"
)

print(
    "7. Figure_GPR_AgingIndex_SHAP_filler.jpeg"
)